Extract
######

In [1]:
import pyodbc
import pandas as pd
import numpy as np


In [2]:
#set source connection
connection_params = (
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=.\sqlexpress22;"
    "DATABASE=BookStore_EG;"
    "TRUSTED_CONNECTION=yes"
)

source_connection = pyodbc.connect(connection_params)

In [3]:
df_book = pd.read_sql_query(f"select * from vbook",source_connection)
df_book.head(20)

C:\Users\abdel\AppData\Local\Temp\ipykernel_22124\70554656.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_book = pd.read_sql_query(f"select * from vbook",source_connection)


,BookID,Title,CategoryDescription,Price
0,1,Chemistry Today 1,Chemistry,72.66
1,2,Islamic Studies 2,Religion,80.34
2,3,Egyptian Law 3,Law,132.41
3,4,Sports in Egypt 4,Sports,102.01
4,5,Geography of Egypt 5,Geography,165.12
5,6,Philosophy Thoughts 6,Philosophy,72.50
6,7,Geography of Egypt 7,Geography,95.98
7,8,Mathematics Basics 8,Mathematics,63.45
8,9,Medical Guide 9,Medicine,214.61
9,10,Economy of Egypt 10,Economics,62.55


In [4]:
df_customer = pd.read_sql_query("select * from Vcustomer", source_connection)
df_customer

C:\Users\abdel\AppData\Local\Temp\ipykernel_22124\3723465119.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customer = pd.read_sql_query("select * from Vcustomer", source_connection)


,CustomerID,full_name,City,ZipCode,State
0,1,Shaker,None,70902,Egypt
1,2,El-Gamal,None,41987,Egypt
2,3,Asmaa,Tanta,62957,Egypt
3,4,Rania,Giza,17634,Egypt
4,5,Nour Ibrahim,None,87592,Egypt
...,...,...,...,...,...
1995,1996,Sara Hassan,None,39585,Egypt
1996,1997,,None,50438,Egypt
1997,1998,El-Naggar,None,24341,Egypt
1998,1999,,None,11133,Egypt


In [5]:
df_order = pd.read_sql_query("select * from Vorder", source_connection)
df_order.head(20)

C:\Users\abdel\AppData\Local\Temp\ipykernel_22124\2182403926.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_order = pd.read_sql_query("select * from Vorder", source_connection)


,OrderID,BookID,CustomerID,OrderDate,Price,Quantity
0,1,198,NaN,2020-12-17,180.70,1.0
1,1,256,NaN,2020-12-17,114.14,2.0
2,1,393,NaN,2020-12-17,188.30,1.0
3,1,492,NaN,2020-12-17,218.49,1.0
4,1,996,NaN,2020-12-17,NaN,1.0
5,2,87,379.0,2020-03-03,150.25,1.0
6,2,538,379.0,2020-03-03,NaN,2.0
7,2,575,379.0,2020-03-03,NaN,3.0
8,3,94,696.0,2020-07-05,85.78,2.0
9,3,270,696.0,2020-07-05,NaN,2.0


In [7]:
df_author = pd.read_sql_query("select * from Vauthor", source_connection)
df_author.head(5)

C:\Users\abdel\AppData\Local\Temp\ipykernel_22124\2949115447.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_author = pd.read_sql_query("select * from Vauthor", source_connection)


,AuthorID,AuthorName
0,1,Naguib Mahfouz
1,2,Youssef Idris
2,3,Taha Hussein
3,4,Abbas Al-Aqqad
4,5,Tawfiq Al-Hakim


In [8]:
source_connection.close()

Transform
######

In [9]:
df_book['Title']=df_book['Title'].str.rsplit(" ", n=1, expand=True)[0]

In [10]:
df_book.sample(20)

,BookID,Title,CategoryDescription,Price
885,886,History of Egypt,History,76.08
174,175,Geography of Egypt,Geography,150.13
674,675,Medical Guide,Medicine,64.53
493,494,Medical Guide,Medicine,64.74
448,449,Science & Nature,Science,80.11
576,577,Agriculture Guide,Agriculture,61.76
802,803,Medical Guide,Medicine,182.99
354,355,Medical Guide,Medicine,140.68
498,499,Geography of Egypt,Geography,165.93
182,183,Economy of Egypt,Economics,126.36


In [11]:
#no missing zipcodes
df_customer['ZipCode'].isna().value_counts()

ZipCode
False    2000
Name: count, dtype: int64

In [12]:
df_customer['City'] = df_customer.groupby('ZipCode')['City'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))


In [13]:
#after  ---> we filled 20 None field
df_customer['City'].isna().value_counts()

City
False    1143
True      857
Name: count, dtype: int64

In [14]:
df_customer['City'].fillna('UNKOWN', inplace=True)
df_customer

,CustomerID,full_name,City,ZipCode,State
0,1,Shaker,UNKOWN,70902,Egypt
1,2,El-Gamal,UNKOWN,41987,Egypt
2,3,Asmaa,Tanta,62957,Egypt
3,4,Rania,Giza,17634,Egypt
4,5,Nour Ibrahim,UNKOWN,87592,Egypt
...,...,...,...,...,...
1995,1996,Sara Hassan,UNKOWN,39585,Egypt
1996,1997,,UNKOWN,50438,Egypt
1997,1998,El-Naggar,Alexandria,24341,Egypt
1998,1999,,UNKOWN,11133,Egypt


In [15]:
avg_price = df_order['Price'].mean().round(2)
df_order['Price'].fillna(avg_price, inplace=True)

avg_quantity = df_order['Quantity'].mean().astype(int)
df_order['Quantity'].fillna(avg_quantity, inplace=True)

df_order['Quantity'] = df_order['Quantity'].astype(int)

customerid = 0
df_order['CustomerID'].fillna(customerid, inplace=True)
df_order['CustomerID'] = df_order['CustomerID'].astype(int)

df_order['OrderDate'] = df_order['OrderDate'].astype('str')
df_order['OrderDate'] = df_order['OrderDate'].str.replace('-','')
df_order['OrderDate'] = df_order['OrderDate'].astype(int)
df_order['OrderDate']


0        20201217
1        20201217
2        20201217
3        20201217
4        20201217
           ...   
32029    20210511
32030    20210511
32031    20220315
32032    20220315
32033    20220315
Name: OrderDate, Length: 32034, dtype: int32

In [16]:
df_order

,OrderID,BookID,CustomerID,OrderDate,Price,Quantity
0,1,198,0,20201217,180.70,1
1,1,256,0,20201217,114.14,2
2,1,393,0,20201217,188.30,1
3,1,492,0,20201217,218.49,1
4,1,996,0,20201217,149.70,1
...,...,...,...,...,...,...
32029,7999,910,790,20210511,142.16,2
32030,7999,951,790,20210511,81.40,2
32031,8000,49,713,20220315,237.53,2
32032,8000,217,713,20220315,78.51,1


In [17]:
df_FactBook = df_order
df_FactBook

,OrderID,BookID,CustomerID,OrderDate,Price,Quantity
0,1,198,0,20201217,180.70,1
1,1,256,0,20201217,114.14,2
2,1,393,0,20201217,188.30,1
3,1,492,0,20201217,218.49,1
4,1,996,0,20201217,149.70,1
...,...,...,...,...,...,...
32029,7999,910,790,20210511,142.16,2
32030,7999,951,790,20210511,81.40,2
32031,8000,49,713,20220315,237.53,2
32032,8000,217,713,20220315,78.51,1


In [18]:
df_book.head(5)

,BookID,Title,CategoryDescription,Price
0,1,Chemistry Today,Chemistry,72.66
1,2,Islamic Studies,Religion,80.34
2,3,Egyptian Law,Law,132.41
3,4,Sports in Egypt,Sports,102.01
4,5,Geography of Egypt,Geography,165.12


In [19]:
df_customer.head(5)

,CustomerID,full_name,City,ZipCode,State
0,1,Shaker,UNKOWN,70902,Egypt
1,2,El-Gamal,UNKOWN,41987,Egypt
2,3,Asmaa,Tanta,62957,Egypt
3,4,Rania,Giza,17634,Egypt
4,5,Nour Ibrahim,UNKOWN,87592,Egypt


In [20]:
df_FactBook.head(5)

,OrderID,BookID,CustomerID,OrderDate,Price,Quantity
0,1,198,0,20201217,180.70,1
1,1,256,0,20201217,114.14,2
2,1,393,0,20201217,188.30,1
3,1,492,0,20201217,218.49,1
4,1,996,0,20201217,149.70,1


Load
######

In [27]:
destination_params=(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=.\sqlexpress22;"
    "DATABASE=BookStore_DWH;"
    "TRUSTED_CONNECTION=yes"
)

destination_connection = pyodbc.connect(destination_params)

cursor = destination_connection.cursor()

In [28]:
df_customer.columns

Index(['CustomerID', 'full_name', 'City', 'ZipCode', 'State'], dtype='object')

In [29]:
for index, row in df_customer.iterrows():
    cursor.execute('insert into Dim_Customer(customer_ID, full_name, city, state, zip_code) values (?, ?, ?, ?, ?)', row['CustomerID'], row['full_name'], row['City'],row['State'] ,row['ZipCode'])
cursor.commit()

In [30]:
df_book.columns

Index(['BookID', 'Title', 'CategoryDescription', 'Price'], dtype='object')

In [32]:
for index, row in df_book.iterrows():
    cursor.execute('insert into Dim_Book(book_id, title, category_name,price) values (?, ?, ?, ?)', row['BookID'], row['Title'], row['CategoryDescription'],row['Price'])
cursor.commit()

In [33]:
df_author.columns

Index(['AuthorID', 'AuthorName'], dtype='object')

In [34]:
for index, row in df_author.iterrows():
    cursor.execute('insert into Dim_Author(author_id, author_name) values (?, ?)', row['AuthorID'], row['AuthorName'])
cursor.commit()

****Fact_Table****

In [35]:
dim_customer = pd.read_sql("SELECT customer_sk, customer_ID FROM Dim_Customer", destination_connection)

dim_customer

C:\Users\abdel\AppData\Local\Temp\ipykernel_22124\1311062180.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_customer = pd.read_sql("SELECT customer_sk, customer_ID FROM Dim_Customer", destination_connection)


,customer_sk,customer_ID
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
1996,1996,1996
1997,1997,1997
1998,1998,1998
1999,1999,1999


In [36]:
df_FactBook = df_FactBook.merge(
    dim_customer[['customer_ID', 'customer_sk']],
    how='left',           
    left_on='CustomerID',  
    right_on='customer_ID' 
)[['OrderID', 'BookID', 'CustomerID', 'OrderDate', 'Price', 'Quantity', 'customer_sk']]

In [37]:
dim_book = pd.read_sql("SELECT book_sk, book_id FROM Dim_Book", destination_connection)

dim_book

C:\Users\abdel\AppData\Local\Temp\ipykernel_22124\4263435852.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_book = pd.read_sql("SELECT book_sk, book_id FROM Dim_Book", destination_connection)


,book_sk,book_id
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
996,996,996
997,997,997
998,998,998
999,999,999


In [38]:
df_FactBook = df_FactBook.merge(
    dim_book[['book_sk', 'book_id']],
    how='left',           
    left_on='BookID',  
    right_on='book_id' 
)[['OrderID', 'BookID', 'CustomerID', 'OrderDate', 'Price', 'Quantity', 'customer_sk', 'book_sk']]

In [39]:
df_FactBook

,OrderID,BookID,CustomerID,OrderDate,Price,Quantity,customer_sk,book_sk
0,1,198,0,20201217,180.70,1,0,198
1,1,256,0,20201217,114.14,2,0,256
2,1,393,0,20201217,188.30,1,0,393
3,1,492,0,20201217,218.49,1,0,492
4,1,996,0,20201217,149.70,1,0,996
...,...,...,...,...,...,...,...,...
32029,7999,910,790,20210511,142.16,2,790,910
32030,7999,951,790,20210511,81.40,2,790,951
32031,8000,49,713,20220315,237.53,2,713,49
32032,8000,217,713,20220315,78.51,1,713,217


In [40]:
df_FactBook.columns

Index(['OrderID', 'BookID', 'CustomerID', 'OrderDate', 'Price', 'Quantity',
       'customer_sk', 'book_sk'],
      dtype='object')

In [41]:
for index, row in df_FactBook.iterrows():
    cursor.execute("""insert into dbo.Fact_Book(date_sk, customer_sk, book_sk, order_id, quantity, unit_price)
                values (?, ?, ?, ?, ?, ?)
                """, row['OrderDate'],row['customer_sk'],row['book_sk'],row['OrderID'],row['Quantity'],row['Price'])
cursor.commit()

In [42]:
destination_connection.close()